# 1. Exploración de Datos — Predicting Irrigation Need

Problema de clasificación multiclase: predecir la necesidad de riego (`Low` / `Medium` / `High`).

Se realiza exploración tabular y visual del dataset de entrenamiento para entender
la distribución de variables, relaciones con el target, y detectar posibles issues de calidad.

**Dataset:** Playground Series S6E4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

continuas = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]

categoricas = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region"
]

TARGET      = "Irrigation_Need"
CLASS_ORDER = ["Low", "Medium", "High"]
PALETTE     = {"Low": "steelblue", "Medium": "goldenrod", "High": "tomato"}

print(f"Train: {train.shape[0]:,} filas x {train.shape[1]} columnas")
print(f"Test:  {test.shape[0]:,} filas x {test.shape[1]} columnas")
print(f"Variables continuas:   {len(continuas)}")
print(f"Variables categoricas: {len(categoricas)}")

In [ ]:
train.head()

## 2. Visión General del Dataset

In [ ]:
train.info()

In [ ]:
train[continuas].describe()

In [ ]:
print("Valores nulos por columna:")
print(train.isnull().sum())
print(f"\nFilas duplicadas (excluyendo id): {train.drop(columns='id').duplicated().sum():,}")

## 3. Distribución del Target

In [ ]:
counts = train[TARGET].value_counts().reindex(CLASS_ORDER)
props  = train[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER)

print(f"{'='*45}")
print("DISTRIBUCIÓN DEL TARGET")
print(f"{'='*45}")
for cls in CLASS_ORDER:
    print(f"  {cls:8s}: {counts[cls]:>7,}  ({props[cls]*100:.2f}%)")
print(f"{'='*45}")

fig, ax = plt.subplots(figsize=(7, 4))
counts.plot.bar(ax=ax, color=[PALETTE[c] for c in CLASS_ORDER], edgecolor="black")
ax.set_title("Distribución del Target (Irrigation Need)", fontsize=13, fontweight="bold")
ax.set_ylabel("Cantidad")
ax.set_xlabel("Irrigation Need")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Variables Continuas

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, col in zip(axes.flat, continuas):
    train[col].hist(bins=50, ax=ax, color="steelblue", edgecolor="black", alpha=0.7)
    ax.set_title(col)
axes.flat[-1].set_visible(False)
plt.suptitle("Distribuciones — Variables continuas", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, col in zip(axes.flat, continuas):
    sns.boxplot(data=train, x=TARGET, y=col, order=CLASS_ORDER,
                palette=PALETTE, ax=ax)
    ax.set_title(col)
axes.flat[-1].set_visible(False)
plt.suptitle("Variables continuas vs Target", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Variables Categóricas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, col in zip(axes.flat, categoricas):
    train[col].value_counts().sort_index().plot.bar(ax=ax, color="steelblue", edgecolor="black")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Distribuciones — Variables categóricas", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, col in zip(axes.flat, categoricas):
    props = (
        train.groupby(col)[TARGET]
             .value_counts(normalize=True)
             .unstack(fill_value=0)
             .reindex(columns=CLASS_ORDER, fill_value=0)
    )
    props.plot.bar(stacked=True, ax=ax,
                   color=[PALETTE[c] for c in CLASS_ORDER], edgecolor="black")
    ax.set_title(col)
    ax.set_ylabel("Proporción")
    ax.tick_params(axis="x", rotation=30)
    ax.legend([], frameon=False)

handles = [plt.Rectangle((0, 0), 1, 1, color=PALETTE[c]) for c in CLASS_ORDER]
fig.legend(handles, CLASS_ORDER, loc="upper right", fontsize=11)
plt.suptitle("Variables categóricas vs Target (proporción)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Correlaciones

In [ ]:
df_corr = train[continuas + [TARGET]].copy()
df_corr[TARGET] = df_corr[TARGET].map({"Low": 0, "Medium": 1, "High": 2})
corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Matriz de correlación", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("Correlación con Irrigation_Need (absoluta, ordenada):")
print(corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).round(4))

## 7. Conclusiones

_Completar después de ejecutar el notebook con los hallazgos principales._